!!! FIRST RUN ONLY

In [ ]:
# 0. CELL — Create venv, install packages, register as Jupyter kernel (First run only)

import platform
import subprocess
import sys
from pathlib import Path

env_name = "od_env"
python_exec = sys.executable
print(f"Detected OS: {platform.system()}")

# 1) Create virtual environment if missing
if not Path(env_name).exists():
    print(f"Creating virtual environment: {env_name}")
    subprocess.run([python_exec, "-m", "venv", env_name], check=True)
else:
    print("Virtual environment already exists.")

# 2) Build correct pip path and python path inside venv
if platform.system() == "Windows":
    venv_python = str(Path(env_name) / "Scripts" / "python.exe")
else:
    venv_python = str(Path(env_name) / "bin" / "python")

# 3) Upgrade pip
print("Upgrading pip inside venv...")
subprocess.run([venv_python, "-m", "pip", "install", "--upgrade", "pip"], check=True)

# 4) Install dependencies into venv (CUDA-enabled torch + helpers)
print("Installing required packages...")
subprocess.run([venv_python, "-m", "pip", "install", 
                "torch", "torchvision", "--extra-index-url", 
                "https://download.pytorch.org/whl/cu121"], check=True)

subprocess.run([venv_python, "-m", "pip", "install",
                "ultralytics", "pycocotools", "albumentations",
                "tensorboard", "opencv-python", "tqdm"], check=True)

# 5) Jupyter kernel registration (THIS WAS MISSING!)
print("Registering this environment as a Jupyter kernel...")
subprocess.run([venv_python, "-m", "pip", "install", "ipykernel"], check=True)
subprocess.run([venv_python, "-m", "ipykernel", "install",
                "--name", env_name, "--display-name", "Python (od_env)"], check=True)

print("="*100)
print("[OK] Environment setup complete.")
print(">>> Restart Jupyter and select kernel: Python (od_env)")
print("="*100)


Imports + CUDA check

In [ ]:
# 1. CELL — Environment & Library Setup (CUDA)

# Core libraries
import os
import random
import numpy as np
import torch

# Vision & detection models (TorchVision)
import torchvision
from torchvision import transforms

# YOLO (Ultralytics)
try:
    from ultralytics import YOLO
except ImportError:
    raise ImportError(
        "[ERROR] Ultralytics is not installed. "
        "Run the environment setup cell or install via: pip install ultralytics"
    )

# COCO tools for annotations and evaluation
try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
except ImportError:
    raise ImportError(
        "[ERROR] pycocotools is not installed. "
        "Run the environment setup cell or install via: pip install pycocotools"
    )

# Data augmentation library
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except ImportError:
    raise ImportError(
        "[ERROR] Albumentations is not installed. "
        "Run the environment setup cell or install via: pip install albumentations"
    )

# OpenCV (image processing / visualization)
try:
    import cv2
except ImportError:
    raise ImportError(
        "[ERROR] OpenCV is not installed. "
        "Run the environment setup cell or install via: pip install opencv-python"
    )

# Matplotlib (notebook visualization)
try:
    import matplotlib.pyplot as plt
except ImportError:
    raise ImportError(
        "[ERROR] Matplotlib is not installed. "
        "Install via: pip install matplotlib"
    )

# Optional: Logging (TensorBoard)
try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:
    print("[WARN] TensorBoard not found. Logging will be disabled.")

# GPU Check
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("Active CUDA device:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

# CUDA REQUIRED for this pipeline
if not torch.cuda.is_available():
    raise EnvironmentError(
        "[ERROR] CUDA was not found. "
        "This training pipeline requires an NVIDIA GPU with CUDA support."
    )

print("[INFO] Environment and libraries loaded successfully.")


Global Config

In [ ]:
# 2. CELL — Global Configuration Settings
from types import SimpleNamespace
from pathlib import Path

CONFIG = SimpleNamespace(

    # MODEL SELECTION
    model_type = "yolo",          # "yolo", "fasterrcnn", "detr"
    model_name = "yolov8n.pt",    # YOLO model variant !!! Just in YOLO

    # DATA SETTINGS
    dataset_root = "/content/data",   # Change to your dataset path
    data_format = "yolo",             # "yolo" (.txt) or "coco" (.json)
    class_names = [],                 # Empty = auto detect or auto-parse later
    num_classes = None,               # If None -> determined automatically
    include_background = True,        # For TorchVision/DETR only, ignored by YOLO

    # TRAINING PARAMETERS
    epochs = 50,
    batch_size = 16,
    learning_rate = 0.001,
    img_size = 640,
    optimizer = "adamw",
    scheduler = "cosine",
    amp = True,
    num_workers = 4,
    random_seed = 42,

    # LOGGING / CHECKPOINTING
    run_name = "experiment_001",      # Change for each experiment
    use_tensorboard = True,

    # BASE OUTPUT DIRECTORY
    base_out_dir = "./runs",
    log_dir = None,
    ckpt_dir = None,
    export_dir = None,

    # EXPORT OPTIONS
    export_onnx = True,    # ONNX required for Orin
    export_trt = True,     # TensorRT engine export
    export_tflite = False, # Not needed for Orin
    export_coreml = False  # Not needed for Orin
)

# Dynamic directory generation
root = Path(CONFIG.base_out_dir) / CONFIG.model_type / CONFIG.run_name
CONFIG.log_dir    = str(root / "logs")
CONFIG.ckpt_dir   = str(root / "checkpoints")
CONFIG.export_dir = str(root / "exports")

for d in [CONFIG.log_dir, CONFIG.ckpt_dir, CONFIG.export_dir]:
    Path(d).mkdir(parents=True, exist_ok=True)

# Data format sanity check
if CONFIG.model_type == "yolo" and CONFIG.data_format != "yolo":
    raise ValueError("YOLO models require data_format='yolo'.")

if CONFIG.model_type != "yolo" and CONFIG.data_format != "coco":
    raise ValueError("FasterRCNN/DETR require data_format='coco'.")

print("[INFO] Global config initialized (directories created)")
print(CONFIG)


Device & Seed Setup

In [ ]:
# 3. CELL — Device & Seed Setup

import torch
import random
import numpy as np

# Select compute device (CUDA required in this project)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

# Safety check — we require CUDA for training
if device.type != "cuda":
    raise EnvironmentError("[ERROR] CUDA device not found. Training requires an NVIDIA GPU.")

# Set random seeds for reproducibility
torch.manual_seed(CONFIG.random_seed)
random.seed(CONFIG.random_seed)
np.random.seed(CONFIG.random_seed)

# If using CUDA, also set deterministic flags
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("[INFO] Seeds set and device ready.")
